# Benchmark and Ablation Study for the Hybrid QNN Corrosion Model

This notebook prints the benchmark tables, evaluates the stored hybrid checkpoint, and runs a controlled ablation study on the same trained model.

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import pennylane as qml
import torch
import torch.nn as nn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    precision_score,
    r2_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:0.4f}')

BASE_DIR = Path('/home/sammarv/quantum_corrosion')
RESULTS_DIR = BASE_DIR / 'results'
MAIN_DIR = RESULTS_DIR / 'qnn_v6_hybrid_amp'
OUT_DIR = RESULTS_DIR / 'benchmark_ablation'
OUT_DIR.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'Primary results directory: {MAIN_DIR}')

def print_block(title: str) -> None:
    print('\n' + '=' * 100)
    print(title)
    print('=' * 100)

def load_json(path: Path) -> dict:
    with open(path, 'r') as f:
        return json.load(f)

def load_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path)

def safe_get(mapping, *keys, default=np.nan):
    for key in keys:
        if isinstance(mapping, dict) and key in mapping:
            return mapping[key]
    return default

def add_row(rows, **kwargs):
    rows.append(kwargs)

Device: cuda
Primary results directory: /home/sammarv/quantum_corrosion/results/qnn_v6_hybrid_amp


In [2]:
# Build primary benchmark directly on 10% train and 10% test splits
X_tr_full = np.load(MAIN_DIR / 'feature_cache' / 'train_X.npy')
y_tr_full = np.load(MAIN_DIR / 'feature_cache' / 'train_y.npy')
X_te_full = np.load(MAIN_DIR / 'feature_cache' / 'test_X.npy')
y_te_full = np.load(MAIN_DIR / 'feature_cache' / 'test_y.npy')

X_tr_10, _, y_tr_10, _ = train_test_split(
    X_tr_full,
    y_tr_full,
    train_size=0.1,
    random_state=42,
    stratify=y_tr_full,
)
X_te_10, _, y_te_10, _ = train_test_split(
    X_te_full,
    y_te_full,
    train_size=0.1,
    random_state=42,
    stratify=y_te_full,
)

print_block('10% TRAIN/TEST SPLITS')
print(f'Train samples: {len(y_tr_10)} / {len(y_tr_full)}')
print(f'Test samples : {len(y_te_10)} / {len(y_te_full)}')

primary_rows = []

# Random Forest baseline
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_tr_10, y_tr_10)
rf_pred = rf.predict(X_te_10)
add_row(
    primary_rows,
    family='Primary benchmark',
    model='rf',
    protocol='train_frac=0.1,test_frac=0.1',
    accuracy=accuracy_score(y_te_10, rf_pred),
    f1=f1_score(y_te_10, rf_pred, average='weighted', zero_division=0),
    precision=precision_score(y_te_10, rf_pred, average='weighted', zero_division=0),
    recall=recall_score(y_te_10, rf_pred, average='weighted', zero_division=0),
    mae=np.nan,
    r2=np.nan,
    source='recomputed on 10% train/test',
)

# Try XGBoost if available
try:
    from xgboost import XGBClassifier

    xgb = XGBClassifier(
        n_estimators=300,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective='multi:softmax',
        eval_metric='mlogloss',
        random_state=42,
    )
    xgb.fit(X_tr_10, y_tr_10)
    xgb_pred = xgb.predict(X_te_10)
    add_row(
        primary_rows,
        family='Primary benchmark',
        model='xgboost',
        protocol='train_frac=0.1,test_frac=0.1',
        accuracy=accuracy_score(y_te_10, xgb_pred),
        f1=f1_score(y_te_10, xgb_pred, average='weighted', zero_division=0),
        precision=precision_score(y_te_10, xgb_pred, average='weighted', zero_division=0),
        recall=recall_score(y_te_10, xgb_pred, average='weighted', zero_division=0),
        mae=np.nan,
        r2=np.nan,
        source='recomputed on 10% train/test',
    )
except Exception as e:
    print(f'XGBoost skipped: {e}')

# Try LightGBM if available
try:
    from lightgbm import LGBMClassifier

    lgbm = LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        random_state=42,
        verbose=-1,
    )
    lgbm.fit(X_tr_10, y_tr_10)
    lgbm_pred = lgbm.predict(X_te_10)
    add_row(
        primary_rows,
        family='Primary benchmark',
        model='lgbm',
        protocol='train_frac=0.1,test_frac=0.1',
        accuracy=accuracy_score(y_te_10, lgbm_pred),
        f1=f1_score(y_te_10, lgbm_pred, average='weighted', zero_division=0),
        precision=precision_score(y_te_10, lgbm_pred, average='weighted', zero_division=0),
        recall=recall_score(y_te_10, lgbm_pred, average='weighted', zero_division=0),
        mae=np.nan,
        r2=np.nan,
        source='recomputed on 10% train/test',
    )
except Exception as e:
    print(f'LightGBM skipped: {e}')

primary_df = pd.DataFrame(primary_rows).sort_values(['accuracy', 'f1'], ascending=False, na_position='last')
print_block('PRIMARY BENCHMARK RESULTS (10% TRAIN / 10% TEST)')
print(primary_df.to_string(index=False))

reference_rows = []

quantum_assisted = load_csv(RESULTS_DIR / 'quantum_assisted_over90_summary.csv')
for _, row in quantum_assisted.iterrows():
    add_row(
        reference_rows,
        family='External reference',
        model=row['model'],
        protocol='separate assisted run',
        accuracy=float(row['accuracy']),
        f1=float(row['f1']),
        precision=safe_get(row.to_dict(), 'precision'),
        recall=safe_get(row.to_dict(), 'recall'),
        mae=np.nan,
        r2=np.nan,
        source='results/quantum_assisted_over90_summary.csv',
    )

pure_search = load_csv(RESULTS_DIR / 'quantum_pure_search_frac10_summary.csv')
for _, row in pure_search.iterrows():
    add_row(
        reference_rows,
        family='External reference',
        model=row['candidate'],
        protocol=f"train_frac={float(row['train_fraction']):0.1f}",
        accuracy=float(row['test_accuracy']),
        f1=float(row['test_f1']),
        precision=np.nan,
        recall=np.nan,
        mae=np.nan,
        r2=np.nan,
        source='results/quantum_pure_search_frac10_summary.csv',
    )

best_over90 = load_csv(RESULTS_DIR / 'best_over90_summary.csv')
best_map = dict(zip(best_over90.iloc[:, 0], best_over90.iloc[:, 1]))
add_row(
    reference_rows,
    family='External reference',
    model=str(best_map.get('model', 'unknown')),
    protocol='separate best-over90 run',
    accuracy=float(best_map.get('accuracy', np.nan)),
    f1=float(best_map.get('f1', np.nan)),
    precision=np.nan,
    recall=np.nan,
    mae=np.nan,
    r2=np.nan,
    source=str(best_map.get('source_checkpoint', 'results/best_over90_summary.csv')),
)

reference_df = pd.DataFrame(reference_rows).sort_values(['accuracy', 'f1'], ascending=False, na_position='last')
print_block('EXTERNAL REFERENCE RESULTS')
print(reference_df.to_string(index=False))

benchmark_df = pd.concat([primary_df, reference_df], ignore_index=True)
benchmark_df.to_csv(OUT_DIR / 'benchmark_results.csv', index=False)
print_block('TOP BENCHMARK MODEL BY ACCURACY')
top_row = benchmark_df.sort_values(['accuracy', 'f1'], ascending=False, na_position='last').iloc[0]
print(top_row.to_string())


10% TRAIN/TEST SPLITS
Train samples: 184 / 1848
Test samples : 123 / 1233


/home/sammarv/.venv/lib/python3.10/site-packages/dask/array/chunk_types.py:110: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy-cuda11x, cupy-cuda12x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  import cupy



PRIMARY BENCHMARK RESULTS (10% TRAIN / 10% TEST)
           family   model                     protocol  accuracy     f1  precision  recall  mae  r2                       source
Primary benchmark xgboost train_frac=0.1,test_frac=0.1    0.8862 0.8815     0.8991  0.8862  NaN NaN recomputed on 10% train/test
Primary benchmark    lgbm train_frac=0.1,test_frac=0.1    0.8780 0.8694     0.9028  0.8780  NaN NaN recomputed on 10% train/test
Primary benchmark      rf train_frac=0.1,test_frac=0.1    0.8537 0.8498     0.8574  0.8537  NaN NaN recomputed on 10% train/test

EXTERNAL REFERENCE RESULTS
            family                  model                 protocol  accuracy     f1  precision  recall  mae  r2                                         source
External reference                unknown separate best-over90 run    0.9724 0.9723        NaN     NaN  NaN NaN           results/models/mobilenetv2_round8.pt
External reference              classical    separate assisted run    0.9724 0.9724     

/home/sammarv/.venv/lib/python3.10/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [3]:
X_tr = np.load(MAIN_DIR / 'feature_cache' / 'train_X.npy')
y_tr = np.load(MAIN_DIR / 'feature_cache' / 'train_y.npy')
X_te = np.load(MAIN_DIR / 'feature_cache' / 'test_X.npy')
y_te = np.load(MAIN_DIR / 'feature_cache' / 'test_y.npy')

# Use 10% for both train and test (stratified for class balance)
X_tr, _, y_tr, _ = train_test_split(
    X_tr,
    y_tr,
    train_size=0.1,
    random_state=42,
    stratify=y_tr,
)
X_te, _, y_te, _ = train_test_split(
    X_te,
    y_te,
    train_size=0.1,
    random_state=42,
    stratify=y_te,
)

print(f'Using 10% train set: {len(y_tr)} samples')
print(f'Using 10% test set : {len(y_te)} samples')

scaler = joblib.load(MAIN_DIR / 'scaler.pkl')
pca = joblib.load(MAIN_DIR / 'pca_16.pkl')

checkpoint = torch.load(MAIN_DIR / 'qnn_best.pt', map_location='cpu')
state = checkpoint.get('state_dict', checkpoint) if isinstance(checkpoint, dict) else checkpoint

N_QLAYERS, N_QUBITS, _ = state['q_weights'].shape
PCA_DIM = state['classical_path.0.weight'].shape[1]
N_CLASSES = state['clf_head.3.bias'].shape[0]

X_te_s = scaler.transform(X_te)
X_te_pca = pca.transform(X_te_s)

if X_te_pca.shape[1] > PCA_DIM:
    X_te_pca = X_te_pca[:, :PCA_DIM]
elif X_te_pca.shape[1] < PCA_DIM:
    pad = PCA_DIM - X_te_pca.shape[1]
    X_te_pca = np.pad(X_te_pca, ((0, 0), (0, pad)), mode='constant')

print_block('CHECKPOINT SHAPE CHECK')
print(f'N_QUBITS  = {N_QUBITS}')
print(f'N_QLAYERS = {N_QLAYERS}')
print(f'PCA_DIM   = {PCA_DIM}')
print(f'N_CLASSES = {N_CLASSES}')
print(f'Test tensor shape after PCA alignment: {X_te_pca.shape}')

dev = qml.device('default.qubit', wires=N_QUBITS)

@qml.qnode(dev, interface='torch', diff_method='backprop')
def vqc(inputs, weights):
    qml.AmplitudeEmbedding(features=inputs, wires=range(N_QUBITS), normalize=True)
    for layer in range(N_QLAYERS):
        for q in range(N_QUBITS):
            qml.Rot(weights[layer, q, 0], weights[layer, q, 1], weights[layer, q, 2], wires=q)
        for q in range(N_QUBITS):
            qml.CNOT(wires=[q, (q + 1) % N_QUBITS])
    return [qml.expval(qml.PauliZ(q)) for q in range(N_QUBITS)]

class HybridQNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.input_scale = nn.Parameter(torch.tensor([np.pi], dtype=torch.float32), requires_grad=False)
        self.q_weights = nn.Parameter(torch.randn((N_QLAYERS, N_QUBITS, 3), dtype=torch.float32) * 0.1)
        self.classical_path = nn.Sequential(nn.Linear(PCA_DIM, 128), nn.ReLU(), nn.Dropout(0.3))
        self.clf_head = nn.Sequential(nn.Linear(N_QUBITS + 128, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, N_CLASSES))
        self.reg_head = nn.Sequential(nn.Linear(N_QUBITS + 128, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x, mode='full'):
        q_results = vqc(x, self.q_weights)
        q_out = torch.stack(q_results, dim=-1).float()
        c_out = self.classical_path(x)

        if mode == 'quantum_only':
            combined = torch.cat([q_out, torch.zeros_like(c_out)], dim=-1)
        elif mode == 'classical_only':
            combined = torch.cat([torch.zeros_like(q_out), c_out], dim=-1)
        else:
            combined = torch.cat([q_out, c_out], dim=-1)

        logits = self.clf_head(combined)
        grams = self.reg_head(combined).squeeze(1)
        return logits, grams

model = HybridQNN().to(device)
model.load_state_dict(state, strict=False)
model.eval()

x_tensor = torch.tensor(X_te_pca, dtype=torch.float32)
y_tensor = torch.tensor(y_te, dtype=torch.long)
loader = DataLoader(TensorDataset(x_tensor, y_tensor), batch_size=64, shuffle=False)

def evaluate_mode(mode='full', keep_dims=None):
    all_pred = []
    all_true = []
    all_gram = []
    for xb, yb in loader:
        xb = xb.to(device)
        if keep_dims is not None and keep_dims < xb.shape[1]:
            xb = xb.clone()
            xb[:, keep_dims:] = 0.0
        with torch.no_grad():
            logits, grams = model(xb, mode=mode)
        all_pred.extend(logits.argmax(dim=1).cpu().numpy())
        all_true.extend(yb.numpy())
        all_gram.extend(grams.cpu().numpy())

    y_true = np.asarray(all_true)
    y_pred = np.asarray(all_pred)
    gram_true = np.asarray([0.5 + 0.5 * int(v) for v in y_true])
    gram_pred = np.asarray(all_gram)

    return {
        'mode': mode,
        'keep_dims': keep_dims if keep_dims is not None else PCA_DIM,
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='weighted', zero_division=0),
        'recall': recall_score(y_true, y_pred, average='weighted', zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f1_weighted': f1_score(y_true, y_pred, average='weighted', zero_division=0),
        'mae': mean_absolute_error(gram_true, gram_pred),
        'r2': r2_score(gram_true, gram_pred),
        'y_true': y_true,
        'y_pred': y_pred,
        'gram_true': gram_true,
        'gram_pred': gram_pred,
    }

full_result = evaluate_mode('full')
print_block('FULL MODEL TEST PERFORMANCE (10% TEST SPLIT)')
print(f"Accuracy            : {full_result['accuracy']:.4f}")
print(f"Balanced Accuracy   : {full_result['balanced_accuracy']:.4f}")
print(f"Weighted Precision  : {full_result['precision']:.4f}")
print(f"Weighted Recall     : {full_result['recall']:.4f}")
print(f"Macro F1            : {full_result['f1_macro']:.4f}")
print(f"Weighted F1         : {full_result['f1_weighted']:.4f}")
print(f"MAE (grams)         : {full_result['mae']:.4f}")
print(f"R^2                 : {full_result['r2']:.4f}")
print('\nClassification report:')
print(classification_report(full_result['y_true'], full_result['y_pred'], target_names=['0.5g', '1.0g', '1.5g', '2.0g', '2.5g']))

cm_full = pd.DataFrame(
    confusion_matrix(full_result['y_true'], full_result['y_pred']),
    index=['0.5g', '1.0g', '1.5g', '2.0g', '2.5g'],
    columns=['0.5g', '1.0g', '1.5g', '2.0g', '2.5g'],
)
print('\nConfusion matrix:')
print(cm_full.to_string())

Using 10% train set: 184 samples
Using 10% test set : 123 samples

CHECKPOINT SHAPE CHECK
N_QUBITS  = 8
N_QLAYERS = 6
PCA_DIM   = 256
N_CLASSES = 5
Test tensor shape after PCA alignment: (123, 256)

FULL MODEL TEST PERFORMANCE (10% TEST SPLIT)
Accuracy            : 0.9350
Balanced Accuracy   : 0.9352
Weighted Precision  : 0.9378
Weighted Recall     : 0.9350
Macro F1            : 0.9341
Weighted F1         : 0.9349
MAE (grams)         : 0.1700
R^2                 : 0.7701

Classification report:
              precision    recall  f1-score   support

        0.5g       0.85      0.96      0.90        24
        1.0g       0.95      0.91      0.93        22
        1.5g       0.92      0.96      0.94        23
        2.0g       1.00      1.00      1.00        27
        2.5g       0.96      0.85      0.90        27

    accuracy                           0.93       123
   macro avg       0.94      0.94      0.93       123
weighted avg       0.94      0.93      0.93       123


Confusion 

In [4]:
ablation_specs = [
    ('full_hybrid', 'full', None),
    ('quantum_only', 'quantum_only', None),
    ('classical_only', 'classical_only', None),
    ('pca_keep_128', 'full', 128),
    ('pca_keep_64', 'full', 64),
    ('pca_keep_32', 'full', 32),
]

ablation_rows = []
for name, mode, keep_dims in ablation_specs:
    result = evaluate_mode(mode=mode, keep_dims=keep_dims)
    ablation_rows.append({
        'condition': name,
        'mode': mode,
        'keep_dims': result['keep_dims'],
        'accuracy': result['accuracy'],
        'balanced_accuracy': result['balanced_accuracy'],
        'precision': result['precision'],
        'recall': result['recall'],
        'f1_macro': result['f1_macro'],
        'f1_weighted': result['f1_weighted'],
        'mae': result['mae'],
        'r2': result['r2'],
    })

ablation_df = pd.DataFrame(ablation_rows)
full_accuracy = float(ablation_df.loc[ablation_df['condition'] == 'full_hybrid', 'accuracy'].iloc[0])
full_f1 = float(ablation_df.loc[ablation_df['condition'] == 'full_hybrid', 'f1_macro'].iloc[0])
ablation_df['delta_accuracy_vs_full'] = ablation_df['accuracy'] - full_accuracy
ablation_df['delta_f1_vs_full'] = ablation_df['f1_macro'] - full_f1
ablation_df = ablation_df.sort_values(['accuracy', 'f1_macro'], ascending=False)

print_block('ABLATION RESULTS')
print(ablation_df.to_string(index=False))

ablation_df.to_csv(OUT_DIR / 'ablation_results.csv', index=False)

print_block('FULL MODEL VS ABLATION DELTAS')
for _, row in ablation_df.iterrows():
    print(
        f"{row['condition']:>15s} | acc={row['accuracy']:.4f} | F1={row['f1_macro']:.4f} | "
        f"Δacc={row['delta_accuracy_vs_full']:+.4f} | ΔF1={row['delta_f1_vs_full']:+.4f} | "
        f"MAE={row['mae']:.4f} | R2={row['r2']:.4f}"
    )


ABLATION RESULTS
     condition           mode  keep_dims  accuracy  balanced_accuracy  precision  recall  f1_macro  f1_weighted    mae     r2  delta_accuracy_vs_full  delta_f1_vs_full
classical_only classical_only        256    0.9512             0.9507     0.9512  0.9512    0.9507       0.9512 0.2300 0.7518                  0.0163            0.0167
   full_hybrid           full        256    0.9350             0.9352     0.9378  0.9350    0.9341       0.9349 0.1700 0.7701                  0.0000            0.0000
  pca_keep_128           full        128    0.9024             0.9061     0.9252  0.9024    0.9015       0.9008 0.2288 0.6688                 -0.0325           -0.0325
   pca_keep_64           full         64    0.8537             0.8616     0.9034  0.8537    0.8486       0.8446 0.2872 0.5654                 -0.0813           -0.0855
   pca_keep_32           full         32    0.8293             0.8411     0.8988  0.8293    0.8169       0.8092 0.3418 0.4904                 

In [5]:
benchmark_df.to_csv(OUT_DIR / 'benchmark_results.csv', index=False)
primary_df.to_csv(OUT_DIR / 'primary_benchmark.csv', index=False)
reference_df.to_csv(OUT_DIR / 'reference_results.csv', index=False)

print_block('SAVED TABLES')
print(f"Benchmark table  : {OUT_DIR / 'benchmark_results.csv'}")
print(f"Primary benchmark: {OUT_DIR / 'primary_benchmark.csv'}")
print(f"Reference table  : {OUT_DIR / 'reference_results.csv'}")
print(f"Ablation table   : {OUT_DIR / 'ablation_results.csv'}")


SAVED TABLES
Benchmark table  : /home/sammarv/quantum_corrosion/results/benchmark_ablation/benchmark_results.csv
Primary benchmark: /home/sammarv/quantum_corrosion/results/benchmark_ablation/primary_benchmark.csv
Reference table  : /home/sammarv/quantum_corrosion/results/benchmark_ablation/reference_results.csv
Ablation table   : /home/sammarv/quantum_corrosion/results/benchmark_ablation/ablation_results.csv


In [6]:
print_block('CLEAN SUMMARY TABLE - Accuracy, Precision, Recall, F1')

# Extract rows with complete metrics
clean_summary_rows = []

# Add ablation results (these have complete metrics)
for _, row in ablation_df.iterrows():
    clean_summary_rows.append({
        'Model': row['condition'].upper(),
        'Type': 'Ablation',
        'Accuracy': row['accuracy'],
        'Precision': row['precision'],
        'Recall': row['recall'],
        'F1': row['f1_macro'],
    })

# Add classical models (RF, XGBoost, LGBM, QNN) from primary benchmark
for _, row in primary_df.iterrows():
    clean_summary_rows.append({
        'Model': row['model'].upper(),
        'Type': 'Classical' if row['model'] in ['rf', 'xgboost', 'lgbm'] else 'Quantum',
        'Accuracy': row['accuracy'],
        'Precision': row['precision'] if not pd.isna(row['precision']) else np.nan,
        'Recall': row['recall'] if not pd.isna(row['recall']) else np.nan,
        'F1': row['f1'],
    })

clean_df = pd.DataFrame(clean_summary_rows).sort_values('Accuracy', ascending=False).reset_index(drop=True)

# Pretty print
print(f"\n{'Model':<25} {'Type':<12} {'Accuracy':>12} {'Precision':>12} {'Recall':>12} {'F1':>12}")
print('-' * 90)
for idx, row in clean_df.iterrows():
    acc_str = f"{row['Accuracy']:>12.4f}"
    prec_str = f"{row['Precision']:>12.4f}" if not pd.isna(row['Precision']) else f"{'N/A':>12s}"
    rec_str = f"{row['Recall']:>12.4f}" if not pd.isna(row['Recall']) else f"{'N/A':>12s}"
    f1_str = f"{row['F1']:>12.4f}"
    print(f"{row['Model']:<25} {row['Type']:<12} {acc_str} {prec_str} {rec_str} {f1_str}")

# Save clean table
clean_df.to_csv(OUT_DIR / 'clean_summary_table.csv', index=False)
print(f"\n✓ Clean summary table saved to {OUT_DIR / 'clean_summary_table.csv'}")


CLEAN SUMMARY TABLE - Accuracy, Precision, Recall, F1

Model                     Type             Accuracy    Precision       Recall           F1
------------------------------------------------------------------------------------------
CLASSICAL_ONLY            Ablation           0.9512       0.9512       0.9512       0.9507
FULL_HYBRID               Ablation           0.9350       0.9378       0.9350       0.9341
PCA_KEEP_128              Ablation           0.9024       0.9252       0.9024       0.9015
XGBOOST                   Classical          0.8862       0.8991       0.8862       0.8815
LGBM                      Classical          0.8780       0.9028       0.8780       0.8694
RF                        Classical          0.8537       0.8574       0.8537       0.8498
PCA_KEEP_64               Ablation           0.8537       0.9034       0.8537       0.8486
PCA_KEEP_32               Ablation           0.8293       0.8988       0.8293       0.8169
QUANTUM_ONLY              Ablation